# EngageMetrics Workforce Analytics

## Overview 
A leading employee engagement analytics company, has just received datasets from multiple sources that need to be consolidated for an urgent executive presentation. The data includes employee feedback from regional offices in CSV format, educational achievement records in Excel spreadsheets, and compensation benchmarks from an external API. The leadership team needs insights on employee engagement trends.

Two different EngageMetrics data sources and an external API from Wikipedia:
- <b>education_data.xlsx:</b> Contains employee educational backgrounds including graduation years and fields of study
- <b>employee_insights.csv:</b> Contains employee performance metrics, satisfaction scores, and work-related information
- <b>Wikipedia REST API:</b> Income statistics in the United States

## Data Loading and Initial Exploration

Import required libraries and load datasets

In [1]:
# Import required libraries

%pip install openpyxl

import pandas as pd
import requests
import json
from datetime import datetime


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Import the quarterly employee engagement feedback data from all regions.  Import and examine the CSV data.

In [2]:
# Import the employee insights data
employee_df = pd.read_csv('DataFiles/employee_insights.csv')

# Display the first few rows
print("\nEmployee Data:")
employee_df.head()


Employee Data:


,employee_id,age,salary,promotion_eligible,last_training_date,department,work_experience,projects_completed,hours_worked_weekly,work_mode,last_promotion_date,satisfaction_score,overtime_hours
0,E0001,54.0,NaN,NaN,15/08/2023,HR,NaN,14.0,NaN,remote work,2022-05-10,NaN,8.4
1,E0002,NaN,$64761,N,15/08/2023,NaN,1 years,NaN,53.3,HYBRID,05-10-2022,NaN,8.1
2,E0003,54.0,NaN,N,15/08/2023,Marketing,8,6.0,32.6,Hybrid,10/05/2022,10.0,5.2
3,E0004,NaN,NaN,No,NaN,NaN,16,1.0,37.8,Remote,05-10-2022,5.0,NaN
4,E0005,29.0,$61486,Y,15/08/2023,NaN,NaN,1.0,53.3,Hybrid,2022-05-10,NaN,0.3


Import and examine the talent development team's educational background data:

In [3]:
# Import the educational background data
education_df = pd.read_excel('DataFiles/education_data.xlsx')

print("Education Data:")
print(education_df.head())


Education Data:
      ID  graduation_year   educational_background
0  E0001             2011               Psychology
1  E0002             1995             Architecture
2  E0003             2007  Business Administration
3  E0004             2000  Business Administration
4  E0005             1991                 Medicine


<b>Tip:</b> Always check your data types and missing values after loading datasets.

## Data Cleaning and Preparation

### Clean the Employee Data

Remove the leading/trailing spaces from each string ad convert each value to lowercase.

In [4]:
employee_df['work_mode'] = employee_df['work_mode'].str.strip().str.lower()


#### Handle Missing Values

First check for missing values.

In [5]:
employee_df.isnull().sum()

employee_id             0
age                    56
salary                 37
promotion_eligible     16
last_training_date     29
department             15
work_experience        29
projects_completed     52
hours_worked_weekly    33
work_mode              16
last_promotion_date    26
satisfaction_score     39
overtime_hours         30
dtype: int64

Replace missing values based on the data type.

In [6]:
# Identify the numeric columns in the employee_df
numeric_cols = employee_df.select_dtypes(include='number')
print(numeric_cols)
print('Numeric columns are :', numeric_cols.columns)

     age  projects_completed  hours_worked_weekly  satisfaction_score  \
0   54.0                14.0                  NaN                 NaN   
1    NaN                 NaN                 53.3                 NaN   
2   54.0                 6.0                 32.6                10.0   
3    NaN                 1.0                 37.8                 5.0   
4   29.0                 1.0                 53.3                 NaN   
..   ...                 ...                  ...                 ...   
95  41.0                10.0                 34.0                 9.0   
96   NaN                 NaN                 43.3                 6.0   
97  26.0                 0.0                  NaN                 6.0   
98   NaN                 NaN                 53.3                 1.0   
99   NaN                 NaN                  NaN                 6.0   

    overtime_hours  
0              8.4  
1              8.1  
2              5.2  
3              NaN  
4              0.3

In [7]:
#  Fill missing numeric values with median
employee_df[numeric_cols.columns] = numeric_cols.fillna(numeric_cols.median())

In [13]:
# Identify the categorical columns in the employee_df
cat_cols = employee_df.select_dtypes(include=['object','string']).columns
print('Categorical columns are :', cat_cols)

Categorical columns are : Index(['employee_id', 'salary', 'promotion_eligible', 'department',
       'work_experience', 'work_mode'],
      dtype='str')


In [14]:
# Fill missing categorical values with mode
employee_df[cat_cols] = employee_df[cat_cols].apply(lambda x: x.fillna(x.mode()[0]))
employee_df[cat_cols].head(10)

,employee_id,salary,promotion_eligible,department,work_experience,work_mode
0,E0001,$104328,Y,HR,16 years,remote work
1,E0002,$64761,N,Finance,1 years,hybrid
2,E0003,$104328,N,Marketing,8,hybrid
3,E0004,$104328,No,Finance,16,remote
4,E0005,$61486,Y,Finance,16 years,hybrid
5,E0006,93128,No,HR,6 years,hybrid
6,E0007,$115377,Y,Finance,11 years,hybrid
7,E0008,$51543,Y,Finance,16 years,hybrid
8,E0009,68755,N,Finance,2,remote work
9,E0010,$104328,Y,Finance,16 years,hybrid


Fill missing date values with the last known valid value.

In [18]:
employee_df[['last_training_date', 'last_promotion_date']] = employee_df[['last_training_date', 'last_promotion_date']].ffill()

Make sure there are no missing values left in the data set.

In [19]:
employee_df.isnull().sum()

employee_id            0
age                    0
salary                 0
promotion_eligible     0
last_training_date     0
department             0
work_experience        0
projects_completed     0
hours_worked_weekly    0
work_mode              0
last_promotion_date    0
satisfaction_score     0
overtime_hours         0
dtype: int64

#### Fix Data Types and Standardize Formats

##### Date Data

In [20]:
# Columns with date info
date_columns = ['last_training_date', 'last_promotion_date']
# Identify inconsistent date formats 
employee_df[date_columns] = employee_df[date_columns].apply(pd.to_datetime, errors='coerce')

# Identify the rows where the conversion failed
inconsistent_rows = employee_df[date_columns].isna()

print("Rows with inconsistent or unparseable formats:")
print(inconsistent_rows)

Rows with inconsistent or unparseable formats:
    last_training_date  last_promotion_date
0                False                False
1                False                False
2                False                False
3                False                False
4                False                False
..                 ...                  ...
95               False                False
96               False                False
97               False                False
98               False                False
99               False                False

[100 rows x 2 columns]


##### Salary Data

In [22]:
# Examine current salary format
print("Salary values sample:")
print(employee_df['salary'].head())

Salary values sample:
0    $104328
1     $64761
2    $104328
3    $104328
4     $61486
Name: salary, dtype: str


Clean the salary column by removing currency symbols and converting to float.

Verify the salary data type.

In [25]:
print(employee_df['salary'].dtype)
print(employee_df['salary'].head())

str
0    $104328
1     $64761
2    $104328
3    $104328
4     $61486
Name: salary, dtype: str


##### Work Mode Data

Examine the work_mode column values.

Standardize the work_mode column values to be consistent case and format.